# 13 Streamlit部署与架构选型

**用途：** 验证Streamlit首屏，并厘清Streamlit、FastAPI和LangSmith不是三选一。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
app_test_code = (
    "import os,tempfile; from pathlib import Path; "
    "t=tempfile.TemporaryDirectory(); r=Path(t.name); "
    "os.environ['CONVERSATION_DB_PATH']=str(r/'conversation.sqlite3'); "
    "os.environ['LANGGRAPH_CHECKPOINT_DB']=str(r/'checkpoint.sqlite3'); "
    "os.environ['HANDOFF_DB_PATH']=str(r/'handoff.sqlite3'); "
    "os.environ['AGENT_MEMORY_DB']=str(r/'memory.sqlite3'); "
    "from streamlit.testing.v1 import AppTest; "
    "at=AppTest.from_file('app.py', default_timeout=60).run(); "
    "print('exceptions='+str(len(at.exception))); "
    "ok=not at.exception; "
    "import agent_graph; agent_graph.CHECKPOINTER.conn.close(); "
    "t.cleanup(); assert ok"
)
app_test = run_command(
    [sys.executable, "-c", app_test_code],
    cwd=PROJECT2_ROOT,
    timeout=120,
)
check("Streamlit AppTest无异常", "exceptions=0" in app_test.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -c import os,tempfile; from pathlib import Path; t=tempfile.TemporaryDirectory(); r=Path(t.name); os.environ['CONVERSATION_DB_PATH']=str(r/'conversation.sqlite3'); os.environ['LANGGRAPH_CHECKPOINT_DB']=str(r/'checkpoint.sqlite3'); os.environ['HANDOFF_DB_PATH']=str(r/'handoff.sqlite3'); os.environ['AGENT_MEMORY_DB']=str(r/'memory.sqlite3'); from streamlit.testing.v1 import AppTest; at=AppTest.from_file('app.py', default_timeout=60).run(); print('exceptions='+str(len(at.exception))); ok=not at.exception; import agent_graph; agent_graph.CHECKPOINTER.conn.close(); t.cleanup(); assert ok
2026-07-28 14:35:04.835 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
exceptions=0
[PASS] Streamlit AppTest无异常


{'检查项': 'Streamlit AppTest无异常', '状态': 'PASS', '说明': ''}

In [3]:
requirements = (DAY1_ROOT / "requirements.txt").read_text(encoding="utf-8").casefold()
rows = [
    {"组件": "Streamlit", "当前": "已实现", "作用": "演示UI、调试台、人工工作台"},
    {"组件": "FastAPI", "当前": "未实现", "作用": "外部API、鉴权、多客户端、前后端解耦"},
    {"组件": "LangSmith", "当前": "未正式接入", "作用": "Trace、线程观察、线上/离线评测"},
]
show_table(rows)
check("当前运行依赖不强制FastAPI", "fastapi" not in requirements)
check("当前运行依赖不强制LangSmith", "langsmith" not in requirements)

,组件,当前,作用
0,Streamlit,已实现,演示UI、调试台、人工工作台
1,FastAPI,未实现,外部API、鉴权、多客户端、前后端解耦
2,LangSmith,未正式接入,Trace、线程观察、线上/离线评测


[PASS] 当前运行依赖不强制FastAPI
[PASS] 当前运行依赖不强制LangSmith


{'检查项': '当前运行依赖不强制LangSmith', '状态': 'PASS', '说明': ''}

## 选型结论

当前继续使用Streamlit，因为作品集需要可演示、可调试、可查看内部JSON和人工工作台。出现微信、移动端、独立前端或第三方系统调用需求时，再抽`AgentService`并增加FastAPI。LangSmith是可选观测平台，不能替代UI或API。

本地SQLite适合单实例演示；Streamlit Community Cloud没有外部持久库时，不能把容器内SQLite当成生产级永久存储。

### 面试官会问

1. 为什么不马上把Streamlit重写成FastAPI？
2. FastAPI最小接口应该有哪些？
3. LangSmith与当前execution trace是什么关系？
4. Cloud重建后checkpoint如何持久化？
5. 生产部署需要哪些鉴权、限流、审计、备份和监控？

### 参考答案

1. **为什么不马上重写FastAPI？** 当前目标是作品集演示、调试内部JSON和人工工作台，Streamlit已经完整覆盖。FastAPI解决的是多客户端、稳定API契约、鉴权和前后端解耦；没有真实调用方时重写会增加工作量，却不直接提升Agent质量。
2. **FastAPI最小接口有哪些？** 至少包括创建/列出会话、发送消息、上传图片、读取运行状态、提交审批、人工接管队列与回复、健康检查；异步长任务还需要任务状态或事件流。身份应由服务端认证上下文注入。
3. **LangSmith和execution trace是什么关系？** 当前trace是业务语义层，记录解析、路由、工具、审批和接管，可本地展示和测试；LangSmith更擅长模型/链路级Trace、数据集和实验管理。二者可以并存，业务trace不应因接入平台而删除。
4. **Cloud重建后怎样持久化？** 容器本地SQLite可能随重部署丢失。生产应把checkpoint、会话、记忆和服务单放到外部Postgres或托管数据库，图片放对象存储，启动时执行迁移并配置备份；Streamlit只保留UI状态。
5. **生产部署还需要什么？** OAuth/JWT和租户授权、API/模型限流、工具权限与审计、PII加密和保留删除、数据库备份恢复、指标/日志/Trace监控、告警、灰度发布、Prompt/模型版本和回滚机制。

**代码落点：** `app.py`、`docs/web_session_architecture.md`、`conversation_repository.py`和各SQLite repository。